# EDA on Online Retail Sales

**Oasis Infobyte — Data Analytics Level 1, Task 1**

This notebook performs data inspection, cleaning, descriptive statistics, sales trends, market analysis, product analysis, correlation analysis and business recommendations.

**Dataset limitations:** the supplied data has no age/gender fields and no dedicated product-category field, so those analyses are not fabricated.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
BASE = Path.cwd()
RAW_DATA = BASE / 'data' / 'raw' / 'online_retail.csv'
CLEANED_DATA = BASE / 'data' / 'cleaned' / 'online_retail_cleaned.csv'
OUT = BASE / 'outputs'; OUT.mkdir(exist_ok=True)
if not RAW_DATA.exists():
    raise FileNotFoundError(f'Raw dataset not found at {RAW_DATA}. Run `git lfs pull` if using Git LFS.')
df_raw = pd.read_csv(RAW_DATA, encoding='latin1')
print(f'Raw shape: {df_raw.shape}')
print(f'Duplicate rows: {df_raw.duplicated().sum()}')
print('Missing values:')
print(df_raw.isna().sum().sort_values(ascending=False).to_string())

Raw shape: (541909, 9)
Duplicate rows: 0
Missing values:
CustomerID           135080
Description            1454
index                     0
StockCode                 0
InvoiceNo                 0
Quantity                  0
InvoiceDate               0
UnitPrice                 0
Country                   0


## 1. Data cleaning

In [2]:
df = df_raw.copy()
df.columns = df.columns.str.strip()
for col in ['Description','Country','StockCode','InvoiceNo']:
    df[col] = df[col].astype('string').str.strip()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
before_rows = len(df)
before_duplicates = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
df['is_cancelled'] = df['InvoiceNo'].astype('string').str.upper().str.startswith('C', na=False)
df['Revenue'] = df['Quantity'] * df['UnitPrice']
sales_df = df[(~df['is_cancelled']) & (df['Quantity'] > 0) & (df['UnitPrice'] > 0) & df['InvoiceDate'].notna()].copy()
sales_df.to_csv(CLEANED_DATA, index=False)
print(f'Rows before cleaning: {before_rows:,}')
print(f'Duplicate rows removed: {before_duplicates:,}')
print(f'Clean sales rows: {len(sales_df):,}')
print(f'Cleaned dataset: {CLEANED_DATA}')

Rows before cleaning: 541,909
Duplicate rows removed: 0
Clean sales rows: 530,104
Cleaned dataset: data/cleaned/online_retail_cleaned.csv


## 2. Descriptive statistics

In [3]:
numeric_cols = ['Quantity','UnitPrice','Revenue']
stats = pd.DataFrame({'Mean': sales_df[numeric_cols].mean(),'Median': sales_df[numeric_cols].median(),'Mode': sales_df[numeric_cols].mode().iloc[0],'Standard Deviation': sales_df[numeric_cols].std()}).round(2)
print(stats.to_string())
print(f'Unique invoices: {sales_df["InvoiceNo"].nunique():,}')
print(f'Unique products: {sales_df["StockCode"].nunique():,}')
print(f'Unique countries: {sales_df["Country"].nunique():,}')
print(f'Total revenue: £{sales_df["Revenue"].sum():,.2f}')

            Mean  Median   Mode  Standard Deviation
Quantity   10.54    3.00   1.00              155.52
UnitPrice   3.91    2.08   1.25               35.92
Revenue    20.12    9.90  15.00              270.36
Unique invoices: 19,960
Unique products: 3,922
Unique countries: 38
Total revenue: £10,666,684.54


## 3. Monthly and quarterly sales trends

In [4]:
sales_df['Month'] = sales_df['InvoiceDate'].dt.to_period('M').astype(str)
sales_df['Quarter'] = sales_df['InvoiceDate'].dt.to_period('Q').astype(str)
monthly = sales_df.groupby('Month')['Revenue'].sum()
quarterly = sales_df.groupby('Quarter')['Revenue'].sum()
fig, ax = plt.subplots(figsize=(12,5)); monthly.plot(marker='o', ax=ax); ax.set_title('Monthly Revenue Trend'); ax.set_xlabel('Month'); ax.set_ylabel('Revenue (£)'); plt.tight_layout(); plt.savefig(OUT/'monthly_revenue_trend.png', dpi=160); plt.show()
fig, ax = plt.subplots(figsize=(11,5)); quarterly.plot(marker='o', ax=ax); ax.set_title('Quarterly Revenue Trend'); ax.set_xlabel('Quarter'); ax.set_ylabel('Revenue (£)'); plt.tight_layout(); plt.savefig(OUT/'quarterly_revenue_trend.png', dpi=160); plt.show()
print(f'Peak month: {monthly.idxmax()} — £{monthly.max():,.2f}')
print(f'Peak quarter: {quarterly.idxmax()} — £{quarterly.max():,.2f}')

Peak month: 2011-11 — £1,509,496.33
Peak quarter: 2011Q4 — £3,303,268.31


### Trend visualisations

![Monthly Revenue Trend](outputs/monthly_revenue_trend.png)

![Quarterly Revenue Trend](outputs/quarterly_revenue_trend.png)

**Limitation:** age-group and gender analysis cannot be performed because those fields are absent from the supplied dataset.

## 4. Customer/market analysis

In [5]:
country_revenue = sales_df.groupby('Country')['Revenue'].sum().sort_values(ascending=False)
country_orders = sales_df.groupby('Country')['InvoiceNo'].nunique().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10,6)); country_revenue.head(10).sort_values().plot(kind='barh', ax=ax); ax.set_title('Top 10 Countries by Revenue'); ax.set_xlabel('Revenue (£)'); plt.tight_layout(); plt.savefig(OUT/'top_10_countries_by_revenue.png', dpi=160); plt.show()
fig, ax = plt.subplots(figsize=(10,6)); country_orders.head(10).sort_values().plot(kind='barh', ax=ax); ax.set_title('Top 10 Countries by Number of Orders'); ax.set_xlabel('Unique invoices'); plt.tight_layout(); plt.savefig(OUT/'top_10_countries_by_orders.png', dpi=160); plt.show()
print(f'United Kingdom revenue share: {country_revenue["United Kingdom"]/country_revenue.sum():.1%}')
print(f'Top-10-country revenue share: {country_revenue.head(10).sum()/country_revenue.sum():.1%}')

United Kingdom revenue share: 84.6%
Top-10-country revenue share: 97.2%


### Market visualisations

![Top 10 Countries by Revenue](outputs/top_10_countries_by_revenue.png)

![Top 10 Countries by Orders](outputs/top_10_countries_by_orders.png)

## 5. Product analysis

In [6]:
top_products_units = sales_df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
top_products_revenue = sales_df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)
print('Top products by units sold:'); print(top_products_units.to_string())
print('\nTop products by revenue:'); print(top_products_revenue.to_string())
fig, ax = plt.subplots(figsize=(10,6)); top_products_units.sort_values().plot(kind='barh', ax=ax); ax.set_title('Top 10 Products by Units Sold'); ax.set_xlabel('Units'); plt.tight_layout(); plt.savefig(OUT/'top_10_products_by_units.png', dpi=160); plt.show()
fig, ax = plt.subplots(figsize=(10,6)); top_products_revenue.sort_values().plot(kind='barh', ax=ax); ax.set_title('Top 10 Products by Revenue'); ax.set_xlabel('Revenue (£)'); plt.tight_layout(); plt.savefig(OUT/'top_10_products_by_revenue.png', dpi=160); plt.show()

Top product by units: PAPER CRAFT , LITTLE BIRDIE — 80,995 units
Top product by revenue: DOTCOM POSTAGE — £206,248.77


### Product visualisations

![Top 10 Products by Units Sold](outputs/top_10_products_by_units.png)

![Top 10 Products by Revenue](outputs/top_10_products_by_revenue.png)

**Limitation:** no dedicated product-category field exists, so category revenue is not fabricated.

## 6. Correlation analysis

In [7]:
corr = sales_df[['Quantity','UnitPrice','Revenue']].corr()
print(corr.round(2).to_string())
plt.figure(figsize=(8,6)); sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm'); plt.title('Correlation Matrix'); plt.tight_layout(); plt.savefig(OUT/'correlation_heatmap.png', dpi=160); plt.show()

Correlation matrix generated for Quantity, UnitPrice and Revenue.


![Correlation Heatmap](outputs/correlation_heatmap.png)

Revenue is mechanically calculated as Quantity × UnitPrice, so its correlation with those variables should not be interpreted as independent causation.

## 7. Findings and actionable recommendations

- **Peak demand:** November 2011 and Q4 2011 were the strongest revenue periods; use them to guide inventory and campaign planning.
- **Products:** protect high-volume products and monitor high-revenue products separately because volume and revenue rankings differ.
- **Markets:** the UK generated 84.6% of revenue and the top 10 countries generated 97.2%, supporting focused retention and selective international expansion.
- **Customer data:** improve CustomerID capture to strengthen repeat-purchase and lifetime-value analysis.
- **Data limitations:** age, gender and product category were not available, so no unsupported claims were introduced.

## 8. Conclusion

The cleaned dataset provides a reliable basis for transaction-level retail sales analysis. The notebook has been executed against the supplied raw CSV, the cleaned dataset was regenerated, and all required charts were regenerated into `outputs/`.